In [1]:
import matplotlib.pyplot as plt

%matplotlib inline

plt.figure(figsize=(20, 5))


<Figure size 2000x500 with 0 Axes>

<Figure size 2000x500 with 0 Axes>

In [2]:
from astropy.io import fits

with fits.open("/Users/wayfinder/Code/fault-in-our-stars/assets/k2-variable-campaigns/lightcurves/campaign-0/hlsp_k2varcat_k2_lightcurve_202059082-c00_kepler_v2_llc.fits") as hdul:
    hdul.info()
    data = hdul[1].data
    print(data.columns.names)


Filename: /Users/wayfinder/Code/fault-in-our-stars/assets/k2-variable-campaigns/lightcurves/campaign-0/hlsp_k2varcat_k2_lightcurve_202059082-c00_kepler_v2_llc.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      67   ()      
  1                1 BinTableHDU     26   1491R x 5C   [D, E, E, E, E]   
['TIME', 'APTFLUX', 'APTFLUX_ERR', 'DETFLUX', 'DETFLUX_ERR']


In [ ]:
from pathlib import Path
import re

import numpy as np
import lightkurve as lk
from astropy.io import fits


def epic_to_9digits(epic):
    return f"{int(epic):09d}"


def find_k2varcat_files(epic, folder):
    """
    Find all K2VarCat llc FITS files for one EPIC in a local folder tree.
    """
    folder = Path(folder)
    epic9 = epic_to_9digits(epic)

    files = sorted(
        p for p in folder.rglob("*.fits")
        if p.name.startswith(f"hlsp_k2varcat_k2_lightcurve_{epic9}-c")
        and p.name.endswith("_llc.fits")
    )
    return files


def _pick_column(data, candidates):
    names = [n.upper() for n in data.columns.names]
    for cand in candidates:
        if cand.upper() in names:
            return data[cand]
    raise KeyError(f"None of these columns were found: {candidates}\nAvailable: {data.columns.names}")


def load_k2varcat_lightcurve(path, use_detrended=True):
    """
    Load one K2VarCat FITS file into a Lightkurve LightCurve.
    The file format description says the first extension is a binary table
    with timestamps, extracted fluxes, and detrended fluxes.
    """
    with fits.open(path) as hdul:
        data = hdul[1].data
        cols = data.columns.names

        # Time column
        time = _pick_column(data, ["TIME", "BJD", "BKJD"])

        # Flux column: prefer detrended if requested, otherwise extracted
        if use_detrended:
            flux = _pick_column(
                data,
                ["DETFLUX", "DETFLUX", "DET_FLUX", "DETREND_FLUX", "DETREND", "FLUX"]
            )
        else:
            flux = _pick_column(
                data,
                ["APTFLUX", "APTFLUX", "APERTURE_FLUX", "FLUX"]
            )

        # Error column is optional
        flux_err = None
        for cand in [
            "DETFLUX_ERR", "DETFLUX_ERR", "DET_FLUX_ERR",
            "APTFLUX_ERR", "APERTURE_FLUX_ERR", "FLUX_ERR"
        ]:
            if cand in cols:
                flux_err = data[cand]
                break

        mask = np.isfinite(time) & np.isfinite(flux)
        if flux_err is not None:
            mask &= np.isfinite(flux_err)

        lc = lk.LightCurve(
            time=time[mask],
            flux=flux[mask],
            flux_err=flux_err[mask] if flux_err is not None else None,
        )

        return lc.remove_nans()


def load_all_k2varcat_lightcurves(epic, folder, use_detrended=True):
    """
    Load all campaign lightcurves for one EPIC.
    """
    files = find_k2varcat_files(epic, folder)

    if not files:
        raise FileNotFoundError(f"No K2VarCat FITS files found for EPIC {epic} in {folder}")

    lcs = [load_k2varcat_lightcurve(file, use_detrended=use_detrended) for file in files]
    return lcs


def stitch_k2varcat_lightcurve(epic, folder, use_detrended=True):
    """
    Stitch all campaign lightcurves for one EPIC into a single light curve.
    """
    lcs = load_all_k2varcat_lightcurves(epic, folder, use_detrended=use_detrended)
    stitched = lk.LightCurveCollection(lcs).stitch().remove_nans()
    return stitched


In [ ]:
epic = 202059070
folder = "/Users/wayfinder/Code/fault-in-our-stars/assets/k2-variable-campaigns/lightcurves/campaign-0"

files = find_k2varcat_files(epic, folder)
print(files)

stitched_lc = stitch_k2varcat_lightcurve(epic, folder, use_detrended=True)
stitched_lc.plot()


---

In [ ]:
!pwd


In [ ]:
import pandas as pd


In [ ]:
K2VARCAT = pd.read_csv(r"../assets/k2-variable-campaigns/hlsp_k2varcat_k2_lightcurve_c00-c04_kepler_v2_catalog.csv")


In [ ]:
K2VARCAT


In [ ]:
# Remove extra spaces from column names
K2VARCAT.columns = K2VARCAT.columns.str.strip()


In [ ]:
K2VARCAT.columns


In [ ]:
C_00 = K2VARCAT[K2VARCAT['Campaign'] == 0]


In [ ]:
C_00


In [ ]:
C_00_folder = "/Users/wayfinder/Code/fault-in-our-stars/assets/k2-variable-campaigns/lightcurves/campaign-0"
epicIDs = C_00['# ID'].unique()

epicIDs


In [ ]:
lcs = {}

for id in epicIDs:
    # lcs[id] = stitch_k2varcat_lightcurve(id, C_00_folder, use_detrended=True)
    lc = stitch_k2varcat_lightcurve(id, C_00_folder, use_detrended=True)

    # Extract time, flux, and flux_err arrays from the LightCurve object
    time = lc.time.value
    flux = lc.flux.value
    flux_err = lc.flux_err.value if 'flux_err' in lc.columns else None
    
    # Store the extracted data in the dictionary with EPIC ID as key
    lcs[id] = {
        'time': time,
        'flux': flux,
        'flux_err': flux_err
    }

# lcs


In [ ]:
# Convert the keys from strings to integers
lcs = {int(id): lcs[id] for id in lcs}

# Now the keys are integers
print(lcs.keys())


In [ ]:
import json 
json.dumps(lcs, indent=4)
